# IndexCalc — Phase 1 & 2 테스트

Phase 1 (IndexSpace, Index, TensorExpr)과 Phase 2 (LaTeX 파서)의 동작을 확인한다.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from indexcalc import IndexSpace, Index, Tensor, IndexRegistry, parse

## 1. IndexSpace & Index 정의

In [ ]:
# 두 가지 index space 정의
spacetime = IndexSpace("spacetime", dim=4, indices="μνλρσ", metric="g")
lorentz   = IndexSpace("lorentz",   dim=4, indices="abcde", metric="η")

print(spacetime)
print(lorentz)

In [ ]:
# Index 생성 & 기본 연산
mu = spacetime.upper("μ")
nu = spacetime.lower("ν")

print(f"mu = {mu}  →  space: {mu.space.name}, position: {mu.position}")
print(f"nu = {nu}  →  space: {nu.space.name}, position: {nu.position}")
print(f"mu.flip() = {mu.flip()}")
print()
print(f"mu contracts with _μ?  {mu.contracts_with(spacetime.lower('μ'))}")
print(f"mu contracts with ^μ?  {mu.contracts_with(spacetime.upper('μ'))}  (같은 위치 — 안 됨)")
print(f"mu contracts with _a?  {mu.contracts_with(lorentz.lower('a'))}  (다른 공간 — 안 됨)")

## 2. Tensor 직접 생성 (Phase 1 방식)

In [ ]:
T = Tensor("T", [spacetime.upper("μ"), spacetime.lower("ν")])
g = Tensor("g", [spacetime.lower("μ"), spacetime.lower("ν")])
V = Tensor("V", [spacetime.upper("μ")])

print(f"T = {T}      rank = {T.rank}")
print(f"g = {g}    rank = {g.rank}")
print(f"V = {V}       rank = {V.rank}")

In [ ]:
# 텐서곱: T^μ_ν * S^ν_λ  →  ν 자동 contraction
S = Tensor("S", [spacetime.upper("ν"), spacetime.lower("λ")])
product = T * S

print(f"T * S = {product}")
print(f"  free indices: {product.free_indices}")
print(f"  contracted:   {product.contracted_pairs}")
print(f"  rank:         {product.rank}")

In [ ]:
# 산술: 합, 차, 스칼라곱
A = Tensor("A", [spacetime.upper("μ"), spacetime.lower("ν")])
B = Tensor("B", [spacetime.upper("μ"), spacetime.lower("ν")])

print(f"A + B = {A + B}")
print(f"A - B = {A - B}")
print(f"3 * A = {3 * A}")
print(f"-A    = {-A}")

## 3. LaTeX 파서 (Phase 2)

In [ ]:
# IndexRegistry에 space 등록
reg = IndexRegistry()
reg.register(spacetime)  # μνλρσ → spacetime
reg.register(lorentz)    # abcde → lorentz

print(f"'μ' → {reg.resolve('μ')}")
print(f"'a' → {reg.resolve('a')}")

In [ ]:
# 단일 텐서 파싱
print(parse("T^{μ}_{ν}", reg))          # T^μ_ν
print(parse("g_{μν}", reg))              # g_μ_ν  (metric)
print(parse("V^μ", reg))                 # V^μ    (중괄호 생략)
print(parse("R^{μ}_{νλρ}", reg))         # R^μ_ν_λ_ρ  (Riemann-like)

In [ ]:
# 텐서곱 + 자동 contraction
expr = parse("T^{μ}_{ν} S^{ν}_{λ}", reg)
print(f"T^μ_ν S^ν_λ  =  {expr}")
print(f"  free: {expr.free_indices}")
print(f"  rank: {expr.rank}")

In [ ]:
# Vielbein: e^a_μ e^b^μ  →  μ 축약, lorentz 인덱스만 남음
expr = parse("e^{a}_{μ} e^{b}^{μ}", reg)
print(f"e^a_μ e^b^μ  =  {expr}")
print(f"  free: {expr.free_indices}  (lorentz만 남음)")

In [ ]:
# 스칼라곱 & 분수
print(parse("2 T^{μ}_{ν}", reg))
print(parse("-T^{μ}_{ν}", reg))
print(parse(r"\frac{1}{2} T^{μ}_{ν}", reg))

In [ ]:
# 합/차
print(parse("A^{μ}_{ν} + B^{μ}_{ν}", reg))
print(parse("A^{μ}_{ν} - B^{μ}_{ν}", reg))

In [ ]:
# 괄호 + 곱
expr = parse("(A^{μ}_{ν} + B^{μ}_{ν}) V^{ν}", reg)
print(f"(A + B) V^ν  =  {expr}")
print(f"  free: {expr.free_indices}")

## 4. 물리학 시나리오 테스트

In [ ]:
# Metric contraction: g_{μν} V^{ν} → V_μ (index lowering)
expr = parse("g_{μν} V^{ν}", reg)
print(f"g_μν V^ν  =  {expr}")
print(f"  free: {expr.free_indices}  →  이것이 V_μ (covector)")

In [ ]:
# 연쇄 contraction: T^μ_ν g^{νλ} S_{λρ}
expr = parse("T^{μ}_{ν} g^{νλ} S_{λρ}", reg)
print(f"T^μ_ν g^νλ S_λρ  =  {expr}")
print(f"  free: {expr.free_indices}  →  (1,1)-tensor")

In [ ]:
# 에러 케이스: free index 개수 불일치
try:
    parse("A^{μ}_{ν} + V^{μ}", reg)
except ValueError as e:
    print(f"예상된 에러: {e}")

In [ ]:
# 에러 케이스: 등록되지 않은 인덱스
try:
    parse("T^{α}", reg)  # α는 spacetime/lorentz 어디에도 등록 안 됨
except KeyError as e:
    print(f"예상된 에러: {e}")